<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "740"
iso_country = "Suriname"
import_portfolio = "h3_ag6_by_sea_M.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h3_ag6_by_sea_DX.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
year = 2023
gdp = 3.470e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Suriname': 

Number of trade records present in dataset: 9244
Number of partner countries present in dataset: 102
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$1450.5m
Total weight within the dataset: 1433.8 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,740,Suriname,702,Singapore,Import,Sea,330610,330610 | Oral or dental hygie,3306,3306 | Oral or dental hygie,33,33 | Essential oils and r,35323,Perfume and toilet preparations,"Petroleum, Chemical and Non-Metallic Mineral P...",33,"CRP - Chemical, rubber, plastic products",63860.0,NaN,63860.0,kg,43014.0,43014.0,13544.70600,19661.0,12216.775231,165472.628772,['Container'],['Container'],864.897163,863.387032,863.534611,863.534611,864.211326,864.211326,904.726413,1603.459053,997.942231,926.496501,929.012656,952.280564,885.886063,1570.342604,1050.397128,962.148076,940.068680,982.644438,869.362148,1707.641142,1302.542849,1139.158025,1053.936803,1024.206529,853.144365,1815.641011,1494.064030,1400.064793,1128.374965,1097.693100,837.232710,1839.897835,1757.859568,1772.973784,1223.275697,1153.091214,821.002102,1802.080680,1726.945406,1892.122438,1399.589929,1174.225602,805.711710,1607.499694,1675.442572,1755.688980,1316.119958,1188.099680,790.120374,1476.625856,1501.028812,1872.339719,1546.815326,1195.967357,35323
1,1,740,Suriname,702,Singapore,Import,Sea,790700,790700 | Zinc; articles n.e.c,7907,7907 | Zinc; articles n.e.c,79,79 | Zinc and articles th,42999,"Metal goods n.e.c. (including anchors, grapnel...",Metal Products,37,FMP - Metal products,5208.2,NaN,5208.2,kg,36.0,36.0,717.16914,19661.0,12216.775231,8761.494186,['Container'],['Container'],45.794833,45.714875,45.722689,45.722689,45.758519,45.758519,47.903724,84.900429,52.839343,49.056414,49.189640,50.421636,46.906160,83.146969,55.616741,50.944104,49.775037,52.029351,46.031247,90.416693,68.967428,60.316479,55.804161,54.229993,45.172543,96.135103,79.108149,74.131049,59.745535,58.120982,44.330048,97.419460,93.075674,93.875946,64.770367,61.054218,43.470665,95.417106,91.438821,100.184664,74.105906,62.173248,42.661064,85.114374,88.711834,92.960745,69.686313,62.907857,41.835530,78.184827,79.476922,99.137203,81.901240,63.324437,42999


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $252.0m (17.4%), ... 

                             cifvalue  vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                           
22 | Beverages, spirits a  37711933.7  66624325.9      2.6      4.6
15 | Animal, vegetable or  26680664.2  25452630.5      1.8      1.8
21 | Miscellaneous edible  23202542.4   6108988.0      1.6      0.4
19 | Preparations of cere  21505621.2   8014779.0      1.5      0.6
02 | Meat and edible meat  19798294.4  10221656.8      1.4      0.7
04 | Dairy produce; birds  17599871.6   6498097.3      1.2      0.5
17 | Sugars and sugar con  16885490.3  31937186.0      1.2      2.2
23 | Food industries, res  15742168.9  22944182.2      1.1      1.6
07 | Vegetables and certa  15663994.4  29913196.7      1.1      2.1
20 | Preparations of vege  15503980.6  10056571.2      1.1      0.7
10 | Cereals               10798750.0  40791305.0      0.7      2.8
16 | Meat, fish

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $259.7m (17.9%), ... 

                     cifvalue   vol_kg_vtw  pct_val  pct_vol
eora_desc                                                   
Food & Beverages  150549915.7  148930057.5     10.4     10.4
Agriculture       107457435.8  134959778.8      7.4      9.4
Fishing             1686978.7     600827.4      0.1      0.0



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $205.3m (14.2%), ... 

                                 cifvalue  vol_kg_vtw  pct_val  pct_vol
gtap_desc                                                              
OFD - Food products n.e.c.     90549177.2  48369528.3      6.2      3.4
VOL - Vegetable oils and fats  34382956.2  42890295.1      2.4      3.0
OMT - Meat products n.e.c.     25325753.5  11527860.8      1.7      0.8
MIL - Dairy products           17778584.6   6524465.3      1.2      0.5
V_F - Vegetables, fruit, nuts  15839713.1  29845053.3      1.1      2.1
SGR - Sugar                    14103010.2  30953832.5      1.0      2.2
GRO - Cereal grains n.e.c.      3832076.8  15782023.7      0.3      1.1
OCR - Crops n.e.c.              1798612.5    604427.7      0.1      0.0
CMT - Bovine meat prods          663412.8    136216.4      0.0      0.0
OSD - Oil seeds                  614152.1    727736.3      0.0      0.1
OAP - Animal products n.e.c.     190913.3     84171.2      0.0      0.

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 17.4% 
GTAP 14.2% 
EORA 17.9%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_16230/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,740,Suriname,702,Singapore,Import,Sea,330610,330610 | Oral or dental hygie,3306,3306 | Oral or dental hygie,33,33 | Essential oils and r,35323,Perfume and toilet preparations,"Petroleum, Chemical and Non-Metallic Mineral P...",33,"CRP - Chemical, rubber, plastic products",63860.0,NaN,63860.0,kg,43014.0,43014.0,13544.70600,19661.0,12216.775231,165472.628772,['Container'],['Container'],864.897163,863.387032,863.534611,863.534611,864.211326,864.211326,904.726413,1603.459053,997.942231,926.496501,929.012656,952.280564,885.886063,1570.342604,1050.397128,962.148076,940.068680,982.644438,869.362148,1707.641142,1302.542849,1139.158025,1053.936803,1024.206529,853.144365,1815.641011,1494.064030,1400.064793,1128.374965,1097.693100,837.232710,1839.897835,1757.859568,1772.973784,1223.275697,1153.091214,821.002102,1802.080680,1726.945406,1892.122438,1399.589929,1174.225602,805.711710,1607.499694,1675.442572,1755.688980,1316.119958,1188.099680,790.120374,1476.625856,1501.028812,1872.339719,1546.815326,1195.967357,35323,NaN
1,1,740,Suriname,702,Singapore,Import,Sea,790700,790700 | Zinc; articles n.e.c,7907,7907 | Zinc; articles n.e.c,79,79 | Zinc and articles th,42999,"Metal goods n.e.c. (including anchors, grapnel...",Metal Products,37,FMP - Metal products,5208.2,NaN,5208.2,kg,36.0,36.0,717.16914,19661.0,12216.775231,8761.494186,['Container'],['Container'],45.794833,45.714875,45.722689,45.722689,45.758519,45.758519,47.903724,84.900429,52.839343,49.056414,49.189640,50.421636,46.906160,83.146969,55.616741,50.944104,49.775037,52.029351,46.031247,90.416693,68.967428,60.316479,55.804161,54.229993,45.172543,96.135103,79.108149,74.131049,59.745535,58.120982,44.330048,97.419460,93.075674,93.875946,64.770367,61.054218,43.470665,95.417106,91.438821,100.184664,74.105906,62.173248,42.661064,85.114374,88.711834,92.960745,69.686313,62.907857,41.835530,78.184827,79.476922,99.137203,81.901240,63.324437,42999,NaN


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $159.8m of $1450.5m (11.0%)
Volume: 174.8 of 1433.8 million kilograms (12.2%)
Value-to-Weight: 0.91 from 1.01 US$ per kg (+-9.6%)
Average distance to market (volume-weighted): 3861.6 from 3057.2 nm (+26.3%)

            food_group  N_com    cifvalue  vol_kg_vtw  pct_val  pct_vol
0    Processed Cereals    327  55032010.8  32965335.9      3.8      2.3
1        Fats and Oils     43  22674543.4  20735906.1      1.6      1.4
2        Meat and Prep     20  18151513.1   9889873.7      1.3      0.7
3           Vegetables    129  17459834.4  31351719.6      1.2      2.2
4                Dairy     71  17379071.0   6397140.3      1.2      0.4
5      Sugar and Honey     56  17149760.3  32012952.2      1.2      2.2
6  Unprocessed Cereals     11  10654576.8  40540876.2      0.7      2.8
7               Fruits     44   1340200.7    879410.8      0.1      0.1



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,14118278.1,24799055.7,15526372.4,14434964.1,14473201.2,14842453.6,13085711.1,28467621.9,27173969.5,27440718.3,19002477.3,17910047.4,12364952.9,22903150.8,23303305.5,28934609.1,23908382.4,18564820.6
1,Fats and Oils,56914539.4,96823750.6,61856013.5,57695229.9,57849674.4,59558236.7,53032470.7,110737868.7,104657390.0,106119716.0,75221972.5,70985154.7,50322687.2,89680375.5,92451358.1,112936269.6,92928728.6,73417229.5
2,Fruits,1850102.0,3242704.3,2033515.9,1892144.7,1897011.0,1944576.4,1715479.8,3726324.1,3559627.4,3595420.9,2487717.5,2344653.2,1621510.1,3001821.5,3049516.8,3781154.0,3125538.6,2430118.1
3,Meat and Prep,16152196.4,28626251.1,17816290.6,16540822.0,16585739.7,17001147.8,14947270.8,32847422.3,31382807.3,31652717.9,21839112.5,20586095.9,14106201.8,26362127.4,26797768.0,33426450.4,27614985.2,21351536.9
4,Processed Cereals,70945187.4,123612526.4,77811973.8,72457375.6,72645830.4,74504295.7,65854934.8,142461319.0,136040056.1,137583152.6,95292733.7,89601271.6,62301808.0,114725343.0,116511125.4,144017773.8,119271080.4,92815030.3
5,Sugar and Honey,42966947.4,72949254.4,46717169.4,43694648.3,43802971.7,44966770.1,40066965.7,84751312.0,80838976.0,82112531.5,57192812.9,53516312.2,38042704.1,68530863.4,69496177.5,84694378.6,70442771.3,55323953.5
6,Unprocessed Cereals,83860995.5,138214607.1,90426622.0,85048282.7,85215996.5,87462158.1,78591341.7,160497683.8,153288568.4,156272036.8,109772075.3,102973898.9,74912988.1,131527334.9,133195394.8,159822764.5,133114400.1,106271937.2
7,Vegetables,91212724.8,160998958.6,100480363.9,93356890.5,93606038.3,95956258.7,84470544.1,184838492.4,176598186.0,178219801.3,123088151.3,116007029.2,79764022.6,148529520.5,150953900.0,187893345.4,155286603.5,120287877.1


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.61,0.08,0.02,0.02,0.04,0.89,0.81,0.83,0.34,0.28,0.61,0.63,0.95,0.66,0.36
1,Fats and Oils,1.76,0.22,0.03,0.04,0.12,2.54,2.28,2.34,0.98,0.79,1.74,1.86,2.76,1.88,1.02
2,Fruits,1.04,0.14,0.03,0.04,0.07,1.50,1.38,1.40,0.58,0.47,1.03,1.07,1.61,1.12,0.60
3,Meat and Prep,0.69,0.09,0.02,0.02,0.05,0.99,0.91,0.92,0.38,0.31,0.68,0.70,1.06,0.74,0.40
4,Processed Cereals,0.96,0.12,0.03,0.03,0.06,1.39,1.28,1.30,0.53,0.43,0.95,0.99,1.48,1.04,0.55
5,Sugar and Honey,1.75,0.22,0.04,0.05,0.12,2.61,2.38,2.45,1.00,0.78,1.78,1.83,2.72,1.89,1.01
6,Unprocessed Cereals,5.10,0.62,0.11,0.13,0.34,7.69,7.01,7.29,2.93,2.29,5.31,5.47,7.97,5.46,2.94
7,Vegetables,4.00,0.53,0.12,0.14,0.27,5.75,5.28,5.37,2.21,1.81,3.94,4.08,6.19,4.33,2.32
